# 실습 10: 3가지 분류 방법 종합 비교 (2교시)

1교시(IQR/RF/XGB) + 2교시(LLM) 결과를 한 화면에서 비교합니다.

**사전 조건**:
- `classification_results.pkl` (1교시 셀 [8]에서 생성)
- `llm_classification_results.pkl` (2교시 셀 [5]에서 생성)


In [ ]:
# %% [Setup] 두 결과 로드
import pickle
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import accuracy_score, f1_score

with open("classification_results.pkl", "rb") as f:
    ml = pickle.load(f)

with open("llm_classification_results.pkl", "rb") as f:
    llm = pickle.load(f)

print(f"1교시 ML 테스트셋: {len(ml['y_test'])}건")
print(f"2교시 LLM 샘플:   {llm['n_samples']}건")


## 1. 정확도 + F1 종합 표

LLM은 100건 샘플에 대한 결과, ML은 전체 테스트셋(2000건) 결과입니다.
샘플 크기 차이 때문에 직접 비교는 참고용으로만.

In [ ]:
# %% [1] 4가지 방법 통합 표
y_test = ml["y_test"]
summary = pd.DataFrame([
    {"방법":"통계 (IQR)",
     "정확도": accuracy_score(y_test, ml["iqr_pred"]),
     "F1":     f1_score(y_test, ml["iqr_pred"]),
     "샘플":   len(y_test),
     "건당 시간(ms)": 0.5,
     "학습 필요":"X", "해석":"임계값"},
    {"방법":"RandomForest",
     "정확도": accuracy_score(y_test, ml["rf_pred"]),
     "F1":     f1_score(y_test, ml["rf_pred"]),
     "샘플":   len(y_test),
     "건당 시간(ms)": 0.3,
     "학습 필요":"O", "해석":"특성 중요도"},
    {"방법":"XGBoost",
     "정확도": accuracy_score(y_test, ml["xgb_pred"]),
     "F1":     f1_score(y_test, ml["xgb_pred"]),
     "샘플":   len(y_test),
     "건당 시간(ms)": 0.5,
     "학습 필요":"O", "해석":"특성 중요도"},
    {"방법":"LLM (Ollama)",
     "정확도": llm["llm_acc"],
     "F1":     llm["llm_f1"],
     "샘플":   llm["n_samples"],
     "건당 시간(ms)": llm["llm_time"]/llm["n_samples"]*1000,
     "학습 필요":"X", "해석":"자연어 *"},
])
print(summary.to_string(index=False))


## 2. 시각화 비교 — 정확도/F1 막대

In [ ]:
# %% [2] 막대 차트
fig = px.bar(
    summary.melt(id_vars="방법", value_vars=["정확도","F1"]),
    x="방법", y="value", color="variable", barmode="group",
    title="3가지 방법 정확도/F1 비교",
    color_discrete_map={"정확도":"#22d3ee","F1":"#a855f7"},
    text_auto=".3f",
)
fig.update_yaxes(range=[0,1])
fig.show()


## 3. 정확도 vs 속도 — 트레이드오프 산점도

In [ ]:
# %% [3] 정확도 vs 속도 (X축 로그)
fig = px.scatter(
    summary, x="건당 시간(ms)", y="정확도", text="방법",
    log_x=True,
    title="정확도 vs 속도 (X축 로그 - LLM이 1000배 느림)",
    color="방법", size_max=22,
)
fig.update_traces(marker_size=18, textposition="top center")
fig.update_yaxes(range=[0.4,1.0])
fig.show()


## 4. 핵심 발견 + 시나리오별 추천

```
시나리오                                  추천 방법
실시간 트래픽 (초당 수천 건)              RandomForest/XGBoost
사전 라벨이 없는 신규 시스템 1차 필터     IQR
의심 요청 정밀 분석 + 근거 생성           LLM
실무 표준 (하이브리드)                    ML 1차 -> LLM 2차
```

**다음**: 3교시 자율 실습 — 이 4가지를 하나의 Streamlit 대시보드로 통합!